# 多元线性回归-向量化

这个 notebook 使用房价预测案例说明向量的定义、向量内积，以及为什么在多元线性回归中通常会使用向量化实现。

## 1. 从一元到多元线性回归

一元线性回归只有 1 个特征，多元线性回归有多个特征。二者的代价函数形式相同，区别主要在预测函数 $f_{\vec{w}, b}(\vec{x})$ 的写法。

| 类型 | 特征 | 预测函数 | 代价函数 |
| --- | --- | --- | --- |
| 一元线性回归 | $x$ | $f_{w,b}(x)=wx+b$ | $J(w,b)=\frac{1}{2m}\sum_{i=1}^{m}(f_{w,b}(x^{(i)})-y^{(i)})^2$ |
| 多元线性回归 | $x_1,x_2,\cdots,x_n$ | $f_{\vec{w},b}(\vec{x})=\vec{w}\cdot\vec{x}+b$ | $J(\vec{w},b)=\frac{1}{2m}\sum_{i=1}^{m}(f_{\vec{w},b}(\vec{x}^{(i)})-y^{(i)})^2$ |

一元线性回归样例，只使用面积 $x$ 预测房价 $y$：

| 编号 | $x$ 面积 | $y$ 房价 |
| --- | --- | --- |
| 1 | 60 | 167.5 |
| ... | ... | ... |

多元线性回归样例，使用面积、卧室数、楼层、房龄 4 个特征预测房价：

| 编号 | $x_1$ 面积 | $x_2$ 卧室 | $x_3$ 楼层 | $x_4$ 房龄 | $y$ 房价 |
| --- | --- | --- | --- | --- | --- |
| 1 | 60 | 2 | 5 | 10 | 167.5 |
| ... | ... | ... | ... | ... | ... |


## 2. 向量的定义

在多元线性回归中，一个样本的多个特征可以组成一个向量。例如，一个房屋样本为：

$$
\vec{x} = [60, 2, 5, 10]
$$

它表示一套面积为 60 平方米、2 个卧室、5 层、房龄 10 年的房子。

模型权重也可以写成向量：

$$
\vec{w} = [2, 15, 1.5, -2]
$$

偏置为：

$$
b = 30
$$

## 3. 向量内积

![向量内积示意图](image/02_vectorization_dot_notrans.gif)

向量内积就是把两个向量对应位置的元素相乘，然后把结果相加：

$$
\vec{w} \cdot \vec{x} = w_1x_1 + w_2x_2 + w_3x_3 + w_4x_4
$$

对房价预测来说：

$$
\hat{y} = \vec{w} \cdot \vec{x} + b
$$

把多元样例首行数据 $\vec{x} = [60, 2, 5, 10]$ 代入：

$$
\hat{y} = 2 \times 60 + 15 \times 2 + 1.5 \times 5 - 2 \times 10 + 30 = 167.5
$$

所以预测房价为 167.5 万元。

## 4. 非向量化实现

非向量化写法使用显式循环：外层循环遍历每个样本，内层循环遍历每个特征，然后逐项计算 $X \cdot \vec{w}$。

In [15]:
def predict_non_vectorized(X, w):
    results = []

    for i in range(len(X)):
        dot_product = 0.0

        for j in range(len(w)):
            dot_product += X[i][j] * w[j]

        results.append(dot_product)

    return results

## 5. 向量化实现

向量化写法把所有样本组成矩阵 $X$，把权重组成向量 $\vec{w}$，然后一次性完成矩阵和向量的乘法：

$$
X\vec{w}
$$

这种写法更接近数学表达式，也能让底层高性能数组库一次性处理大量数据。

In [16]:
import numpy as np


def predict_vectorized(X, w):
    X_array = np.array(X, dtype=float)
    w_array = np.array(w, dtype=float)

    # 等价写法 1：X_array.dot(w_array)
    # 等价写法 2：np.dot(X_array, w_array)
    # 等价写法 3：np.matmul(X_array, w_array)
    return X_array @ w_array

## 6. 耗时对比

下面构造两个很大的向量 $x$ 和 $\vec{w}$，直接调用第 4、5 节的函数，对比非向量化和向量化实现的耗时。

In [17]:
import numpy as np
import time


np.random.seed(1)
w = np.random.rand(10_000_000)
x = np.random.rand(10_000_000)

tic = time.time()
result_loop = predict_non_vectorized([x], w)
toc = time.time()

print(f"Non-vectorized version duration: {1000 * (toc - tic):.4f} ms")

tic = time.time()
result_vectorized = predict_vectorized([x], w)
toc = time.time()

print(f"Vectorized version duration: {1000 * (toc - tic):.4f} ms")
print(f"Results match: {np.allclose(result_loop, result_vectorized)}")

del x
del w

Non-vectorized version duration: 2487.6854 ms
Vectorized version duration: 121.3481 ms
Results match: True


## 7. 向量化的好处

![向量内积示意图](image/02_vectorization_dot_notrans_1.jpg)

向量化通过减少 Python 层循环开销，并利用底层数值计算库对 CPU/GPU 的优化能力，让大规模矩阵和向量运算更高效。